In [1]:
import re
import pandas as pd

In [2]:
data=\
    (
        pd
        .read_csv("uk_churches.csv")
    )

In [8]:
df=data.copy()

keywords = [
"church land",
"estate",
"synagogues",
"education",
"school",
"dioces",
"mosque",
"islam",
"muslim",
" uk",
"uk ",
"guru",
"conference"
]

# Fill NA with empty string to avoid boolean indexing issues
name_series = df["charity_name"].fillna("")

# Generate boolean flags for each keyword
flags = {
    f"has_{kw}": name_series.str.contains(re.escape(kw), case=False, regex=True)
    for kw in keywords
}

df = df.assign(**flags)

# Check that "church" is NOT present in charity_name
no_church = ~df["charity_name"].fillna("").str.contains(r"\bchurch\b", case=False, regex=True)

# Combine: (has_ uk OR has_uk ) AND NOT church
df["has_uk"] = (df["has_ uk"] | df["has_uk "]) & no_church

# Drop the temporary spaced columns
df = df.drop(columns=["has_ uk", "has_uk "])

# 1. Select the flag columns in the exact order you want them represented
bitmap_cols = [c for c in df.columns if c.startswith("has_")]

# 2. Concatenate into a binary string (e.g., '101', '001', '110')
df["bitmap"] = df[bitmap_cols].astype(int).astype(str).agg("".join, axis=1)

# Parse the binary string to an integer
df["bitmap_int"] = df["bitmap"].apply(lambda x: int(x, 2))

In [11]:
df

,charity_name,registered_number,contact_postcode,contact_web,income,expenditure,denomination,principal_bank,reserves,assets_total,...,has_school,has_dioces,has_mosque,has_islam,has_muslim,has_guru,has_conference,has_uk,bitmap,bitmap_int
0,HITCHAM FREE CHURCH,200027,NaN,NaN,93073.0,264019.0,Non‑denominational,Caf Bank,2024499.0,2077693.0,...,False,False,False,False,False,False,False,False,000000000000,0
1,MELLS CHURCH LANDS,200999,NaN,NaN,10232.0,5511.0,Church of England,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,100000000000,2048
2,CHURCH AND TOWN ESTATE (EXCLUDING CHURCH),201239,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,010000000000,1024
3,CHURCH AND POOR LANDS FOR PARISH OF ST JOHN,201444,NaN,NaN,29224.0,13918.0,Church of England,Lloyds,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
4,CHURCH AND POORS ESTATE,201560,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,010000000000,1024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31686,BRIDGE CHURCH FOR LIFE,1218016,BS32 0DA,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
31687,NEW COVENANT CHURCH TRANSFORMING LIVES CITY,1218032,SE28 8QE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
31688,THE PAROCHIAL CHURCH COUNCIL OF THE ECCLESIAST...,1218085,WS14 9LE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
31689,IGLESIA EVANGELICA VIDA Y GRACIA LONDON,1218170,CR4 3AR,NaN,NaN,NaN,Evangelical,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0


In [17]:
(
    df
    .query("bitmap_int == 0")
)

,charity_name,registered_number,contact_postcode,contact_web,income,expenditure,denomination,principal_bank,reserves,assets_total,...,has_school,has_dioces,has_mosque,has_islam,has_muslim,has_guru,has_conference,has_uk,bitmap,bitmap_int
0,HITCHAM FREE CHURCH,200027,NaN,NaN,93073.0,264019.0,Non‑denominational,Caf Bank,2024499.0,2077693.0,...,False,False,False,False,False,False,False,False,000000000000,0
3,CHURCH AND POOR LANDS FOR PARISH OF ST JOHN,201444,NaN,NaN,29224.0,13918.0,Church of England,Lloyds,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
5,CROWLE HACKETT PARISH LANDS CHARITY,201953,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
6,PARISH BANK,202638,NaN,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
7,THE PARISH LANDS CHARITY (IN THE ANCIENT PARIS...,203004,NaN,NaN,47655.0,56806.0,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31685,"CELESTIAL CHURCH OF CHRIST, LIGHT INTERNATIONAL",1218049,N16 5DU,NaN,NaN,NaN,Church of Christ,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
31686,BRIDGE CHURCH FOR LIFE,1218016,BS32 0DA,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
31687,NEW COVENANT CHURCH TRANSFORMING LIVES CITY,1218032,SE28 8QE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
31688,THE PAROCHIAL CHURCH COUNCIL OF THE ECCLESIAST...,1218085,WS14 9LE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,000000000000,0
